# Clean Mario behavioral data

This notebook keeps the recorded behavioral fields (stimuli, choices, response times, values, conditions, and source-row provenance). It does **not** create mushroom/stimulus ID columns or require `stim72_mapping.csv`.

Valid behavioral trials are exported by phase and as one combined file. Technical event rows and source-recorded bug rows are retained in separate audit files.

In [1]:
from pathlib import Path
import csv
import numpy as np
import pandas as pd

# Locate the folder containing this notebook.
try:
    import ipynbname
    base_dir = Path(ipynbname.path()).parent
except Exception:
    base_dir = Path.cwd()

print("Using folder:", base_dir)

# Exclude mappings and any outputs if they are ever copied beside the raw files.
EXCLUDE_FILES = {
    "stim72_mapping.csv",
    "cleaned_behavioral_data.csv",
    "cleaned_exploration_phase_data.csv",
    "cleaned_odd_one_out_data.csv",
    "cleaned_memory_phase_data.csv",
    "cleaning_event_log.csv",
    "source_bugged_trials.csv",
}

csv_paths = sorted(p for p in base_dir.glob("*.csv") if p.name not in EXCLUDE_FILES)
if not csv_paths:
    raise FileNotFoundError(f"No input CSV files found in {base_dir}")

print(f"Found {len(csv_paths)} input CSV(s).")

def read_csv_flexible(path):
    """Read task exports whose survey rows can be wider than the first header row."""
    with path.open(newline="", encoding="utf-8", errors="replace") as handle:
        rows = list(csv.reader(handle))

    if not rows:
        return pd.DataFrame()

    header = rows[0]
    width = len(header)
    fixed_rows = []
    short_rows = 0
    long_rows = 0

    for row in rows[1:]:
        if len(row) < width:
            row = row + [""] * (width - len(row))
            short_rows += 1
        elif len(row) > width:
            # The over-wide rows in these exports are survey records. Behavioral
            # fields occur within the original header width.
            row = row[:width]
            long_rows += 1
        fixed_rows.append(row)

    frame = pd.DataFrame(fixed_rows, columns=header)
    frame = frame.replace("", pd.NA).convert_dtypes()
    if short_rows or long_rows:
        print(f"  {path.name}: normalized {short_rows} short row(s), truncated {long_rows} survey row(s).")
    return frame

frames = []
for path in csv_paths:
    frame = read_csv_flexible(path)
    frame["__source_file__"] = path.name
    frame["__row_in_file__"] = np.arange(len(frame))
    frames.append(frame)

df_all = pd.concat(frames, ignore_index=True, sort=False)
if "trial_type" not in df_all.columns:
    raise KeyError("Expected a 'trial_type' column in the task exports.")

print("Combined raw shape:", df_all.shape)
print(df_all["trial_type"].value_counts(dropna=False).head(20).to_string())

Using folder: /Users/jianleguo/Documents/GitHub/vmps_mario/data/batch02_new_set_9_participant/clean_use/complete_only_csvs
Found 48 input CSV(s).
  data_5bbc0004ac45e6000131d380.csv: normalized 0 short row(s), truncated 4 survey row(s).
  data_5c476621b521ee00019abf67.csv: normalized 0 short row(s), truncated 4 survey row(s).
  data_5c58e01e6f4ec5000164c35a.csv: normalized 0 short row(s), truncated 4 survey row(s).
  data_5e6b034bffc1a90c3fe8c234.csv: normalized 0 short row(s), truncated 4 survey row(s).
  data_5e84bd7c34d5d9072128fac1.csv: normalized 0 short row(s), truncated 4 survey row(s).
  data_5ee92b925f0f482bb1e73668.csv: normalized 0 short row(s), truncated 4 survey row(s).
  data_5ef6308c40846d2dfca0b66a.csv: normalized 0 short row(s), truncated 4 survey row(s).
  data_615ddab1e4f013092538b6c5.csv: normalized 0 short row(s), truncated 4 survey row(s).
  data_62877d5f6a054e53313ac916.csv: normalized 0 short row(s), truncated 4 survey row(s).
  data_62c84ca82b03c9aedbca4a2a.csv

In [2]:
# Assign phases while preserving every original behavioral column.
PARTICIPANT_COL = "id" if "id" in df_all.columns else "__source_file__"
EXPLORE_TYPES = {"explore_decision", "room_choice", "explore_seen"}
MEMORY_TYPES = {"memory_choice", "oldnew_response"}
OOO_TYPE = "odd_one_out"
BUG_TYPE = "bugged_trial"

def assign_phases_one_participant(group):
    group = group.copy()
    group["__time_sort__"] = pd.to_numeric(group.get("time_elapsed"), errors="coerce")
    group = group.sort_values(
        ["__time_sort__", "__row_in_file__"],
        kind="mergesort",
        na_position="last",
    )
    group["__position__"] = np.arange(len(group))

    explore_positions = group.loc[group["trial_type"].isin(EXPLORE_TYPES), "__position__"]
    memory_positions = group.loc[group["trial_type"].isin(MEMORY_TYPES), "__position__"]

    explore_start = int(explore_positions.min()) if len(explore_positions) else np.inf
    explore_end = int(explore_positions.max()) if len(explore_positions) else -np.inf
    memory_end = int(memory_positions.max()) if len(memory_positions) else -np.inf
    second_ooo_boundary = memory_end if len(memory_positions) else explore_end

    group["phase"] = pd.Series("other", index=group.index, dtype="string")
    group.loc[group["trial_type"].eq(OOO_TYPE), "phase"] = "odd_one_out"
    group.loc[group["trial_type"].isin(EXPLORE_TYPES), "phase"] = "exploration"
    group.loc[group["trial_type"].isin(MEMORY_TYPES), "phase"] = "memory"
    group.loc[group["trial_type"].eq(BUG_TYPE), "phase"] = "source_bug"

    group["ooo_block"] = pd.Series(pd.NA, index=group.index, dtype="Int64")
    is_ooo = group["trial_type"].eq(OOO_TYPE)
    group.loc[is_ooo & group["__position__"].lt(explore_start), "ooo_block"] = 1
    group.loc[is_ooo & group["__position__"].gt(second_ooo_boundary), "ooo_block"] = 2

    return group.drop(columns=["__time_sort__", "__position__"])

tagged_parts = []
for _, participant_rows in df_all.groupby(PARTICIPANT_COL, sort=False, dropna=False):
    tagged_parts.append(assign_phases_one_participant(participant_rows))
df_tagged = pd.concat(tagged_parts, ignore_index=True, sort=False)

# Technical event rows (for example, too_fast) are logs rather than completed trials.
if "event" in df_tagged.columns:
    df_event_log = df_tagged[df_tagged["event"].notna()].copy()
else:
    df_event_log = df_tagged.iloc[0:0].copy()

df_bugged = df_tagged[df_tagged["trial_type"].eq(BUG_TYPE)].copy()

def finalize_phase(frame, remove_event_rows=True):
    frame = frame.copy()
    if remove_event_rows and "event" in frame.columns:
        frame = frame[frame["event"].isna()].copy()

    # No derived mushroom/stimulus IDs are retained. Participant `id` is kept.
    derived_id_columns = [col for col in frame.columns if "stim72_id" in col.lower()]
    frame = frame.drop(columns=derived_id_columns, errors="ignore")
    frame = frame.dropna(axis=1, how="all")

    if "time_elapsed" in frame.columns:
        frame["__time_sort_output__"] = pd.to_numeric(frame["time_elapsed"], errors="coerce")
    sort_columns = [col for col in ["__source_file__", "__time_sort_output__", "__row_in_file__"] if col in frame.columns]
    if sort_columns:
        frame = frame.sort_values(sort_columns, kind="mergesort", na_position="last")
    frame = frame.drop(columns=["__time_sort_output__"], errors="ignore")
    return frame.reset_index(drop=True)

df_odd = finalize_phase(df_tagged[df_tagged["phase"].eq("odd_one_out")])
df_expl = finalize_phase(df_tagged[df_tagged["phase"].eq("exploration")])
df_mem = finalize_phase(df_tagged[df_tagged["phase"].eq("memory")])
df_event_log = finalize_phase(df_event_log, remove_event_rows=False)
df_bugged = finalize_phase(df_bugged)

# Keep the original trial_index. Add a separate OOO phase label.
if len(df_odd):
    df_odd["ooo_phase"] = df_odd["ooo_block"].map({1: "OOO_1", 2: "OOO_2"}).fillna("OOO_unknown")

df_behavioral = pd.concat([df_odd, df_expl, df_mem], ignore_index=True, sort=False)
df_behavioral = df_behavioral.dropna(axis=1, how="all")
df_behavioral["__time_sort_output__"] = pd.to_numeric(df_behavioral["time_elapsed"], errors="coerce")
df_behavioral = df_behavioral.sort_values(
    ["__source_file__", "__time_sort_output__", "__row_in_file__"],
    kind="mergesort",
    na_position="last",
).drop(columns=["__time_sort_output__"]).reset_index(drop=True)

In [3]:
# Quality checks before export.
print("Participants:", df_behavioral[PARTICIPANT_COL].nunique(dropna=True))
print("Valid behavioral rows:", len(df_behavioral))
print("Technical event-log rows removed from trial files:", len(df_event_log))
print("Source bug rows kept for audit:", len(df_bugged))

print("\nRows by phase and trial type:")
print(
    df_behavioral.groupby(["phase", "trial_type"], dropna=False)
                 .size()
                 .rename("rows")
                 .to_string()
)

print("\nRows per participant and phase:")
print(pd.crosstab(df_behavioral[PARTICIPANT_COL], df_behavioral["phase"]).to_string())

stim_id_columns = [col for col in df_behavioral.columns if "stim72_id" in col.lower()]
assert not stim_id_columns, f"Unexpected derived stimulus-ID columns: {stim_id_columns}"
assert not ("event" in df_behavioral.columns and df_behavioral["event"].notna().any())

print("\nBehavioral columns retained:")
print(df_behavioral.columns.tolist())

Participants: 48
Valid behavioral rows: 62145
Technical event-log rows removed from trial files: 41
Source bug rows kept for audit: 433

Rows by phase and trial type:
phase        trial_type      
exploration  explore_decision    24867
             explore_seen        24090
             room_choice          1055
memory       memory_choice        2533
odd_one_out  odd_one_out          9600

Rows per participant and phase:
phase                                        exploration  memory       odd_one_out
id                                                                                
5bbc0004ac45e6000131d380                            1118           60          200
5c476621b521ee00019abf67                            1203           60          200
5c58e01e6f4ec5000164c35a                             764           60          200
5e6b034bffc1a90c3fe8c234                            1195           60          200
5e84bd7c34d5d9072128fac1                             884           60        

In [4]:
# Export one combined clean file plus convenient phase-specific files.
out_dir = base_dir / "cleaned_outputs"
out_dir.mkdir(parents=True, exist_ok=True)

outputs = {
    "cleaned_behavioral_data.csv": df_behavioral,
    "cleaned_exploration_phase_data.csv": df_expl,
    "cleaned_odd_one_out_data.csv": df_odd,
    "cleaned_memory_phase_data.csv": df_mem,
    "cleaning_event_log.csv": df_event_log,
    "source_bugged_trials.csv": df_bugged,
}

for filename, frame in outputs.items():
    frame.to_csv(out_dir / filename, index=False)
    print(f"Saved {filename}: {len(frame):,} rows x {len(frame.columns):,} columns")

print("Output folder:", out_dir)

Saved cleaned_behavioral_data.csv: 62,145 rows x 85 columns
Saved cleaned_exploration_phase_data.csv: 50,012 rows x 23 columns
Saved cleaned_odd_one_out_data.csv: 9,600 rows x 15 columns
Saved cleaned_memory_phase_data.csv: 2,533 rows x 63 columns
Saved cleaning_event_log.csv: 41 rows x 11 columns
Saved source_bugged_trials.csv: 433 rows x 24 columns
Output folder: /Users/jianleguo/Documents/GitHub/vmps_mario/data/batch02_new_set_9_participant/clean_use/complete_only_csvs/cleaned_outputs
